In [0]:
%sql
CREATE OR REPLACE TEMP VIEW bronze_purchase_order AS
SELECT *
FROM `operations-catalog`.supplychain.bronze_purchase_order;

In [0]:
%sql
SELECT * FROM bronze_purchase_order
LIMIT 5;

In [0]:

%sql
CREATE OR REPLACE TEMP VIEW bronze_purchase_order_parsed AS
SELECT
    TRY_CAST(key AS STRING) AS key,
    TRY_CAST(value AS STRING) AS value,
    *
    EXCEPT (key, value)
FROM bronze_purchase_order;

In [0]:
%sql
SELECT *
FROM bronze_purchase_order_parsed
LIMIT 5;

In [0]:
%sql
DESCRIBE bronze_purchase_order_parsed;

In [0]:
from pyspark.sql.functions import from_json, parse_json
from pyspark.sql.types import *

key_schema = StructType([
    StructField(
        "schema",
        StructType([
            StructField("type", StringType()),
            StructField(
                "fields",
                ArrayType(
                    StructType([
                        StructField("type", StringType()),
                        StructField("optional", BooleanType()),
                        StructField("field", StringType())
                    ])
                )
            ),
            StructField("optional", BooleanType()),
            StructField("name", StringType())
        ])
    ),
    StructField(
        "payload",
        StructType([
            StructField("PO_ID", StringType())
        ])
    )
])

bronze_purchase_order_parsed = spark.table("bronze_purchase_order_parsed")
bronze_purchase_order_casted = bronze_purchase_order_parsed.withColumn(
    "key",
    from_json("key", key_schema)
).withColumn(
    "value",
    parse_json("value")
)

bronze_purchase_order_casted.show(5)

In [0]:
bronze_purchase_order_casted.printSchema()

In [0]:
bronze_purchase_order_parsed.printSchema()

In [0]:
bronze_purchase_order_casted.createOrReplaceTempView("bronze_purchase_order_casted")

In [0]:
%sql
SELECT key.payload.PO_ID as PO_ID,
value:payload:SupplierID as SupplierID,
value:payload:OrderDate as OrderDate,
value:payload:ExpectedDeliveryDate as ExpectedDeliveryDate,
value:payload:Status as Status,
value:payload:TotalCost as TotalCost,
value:payload:__op as Operation,
value:payload:__deleted as Deleted,
*
EXCEPT (key, value)
FROM bronze_purchase_order_casted
LIMIT 5;

In [0]:
pe":"string","optional":true,"field":"OrderDate"},{"type":"string","optional":true,"field":"Expected...
{"schema":{"type":"struct","fields":[{"type":"string","optional":false,"field":"PO_ID"}],"optional":false,"name":"supplychain.public.purchase_order.Key"},"payload":{"PO_ID":"PO-8088"}}
{"schema":{"type":"struct","fields":[{"type":"string","optional":false,"field":"PO_ID"},{"type":"string","optional":true,"field":"SupplierID"},{"type":"string","optional":true,"field":"OrderDate"},{"type":"string","optional":true,"field":"ExpectedDeliveryDate"},{"type":"string","optional":true,"field":"Status"},{"type":"string","optional":true,"field":"TotalCost"},{"type":"string","optional":true,"field":"__deleted"},{"type":"string","optional":true,"field":"__op"},{"type":"string","optional":true,"field":"__table"},{"type":"int64","optional":true,"field":"__lsn"},{"type":"int64","optional":true,"field":"__source_ts_ms"}],"optional":false,"name":"supplychain.public.purchase_order.Value"},"payload":{"PO_ID":"PO-8088","SupplierID":"SUP04","OrderDate":"2023-12-06","ExpectedDeliveryDate":"2024-01-16","Status":"#N/A","TotalCost":"18978.00","__deleted":"false","__op":"r","__table":"purchase_order","__lsn":90951344,"__source_ts_ms":1789791513517}}